In [1]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()
key = os.getenv("GEMINI_API_KEY")
print("Key found:", key is not None)

client = genai.Client(api_key=key)
MODEL = "gemini-flash-latest"   # same model as Project 2

reply = client.models.generate_content(model=MODEL, contents="Reply with the single word OK.")
print("Gemini says:", reply.text)

Key found: True
Gemini says: OK


In [2]:
import duckdb, json, re, time

con = duckdb.connect()
con.execute("CREATE OR REPLACE VIEW apps AS SELECT * FROM 'data/powerbi/apps.csv'")
con.execute("CREATE OR REPLACE VIEW waits AS SELECT * FROM 'data/powerbi/waits.csv'")

SCHEMA = """
Table apps: one row per loan application (31,509 rows, all started in 2016).
- app_id (text)
- loan_goal (text), e.g. 'Car', 'Home improvement'
- app_type (text): 'New credit' or 'Limit raise'
- requested_amount (number; 0 means not given)
- start_time (timestamp)
- online (1 = submitted online, 0 = entered by staff)
- had_incomplete (1 if documents were ever missing, else 0)
- offers (number of offers made)
- outcome (text): 'A_Pending' means approved, 'A_Cancelled', 'A_Denied', 'Open' means no outcome yet
- days_to_outcome (days from start to outcome; empty if Open)

Table waits: one row per move between application steps, finished applications only.
- app_id, step, next_step (text, e.g. 'A_Complete', 'A_Validating'), wait_days (number)

Rule: approval, cancellation and denial rates use finished applications only (outcome <> 'Open').
"""

CACHE_FILE = "data/ai_cache.json"
cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}

def ask_gemini(prompt):
    if prompt in cache:
        return cache[prompt]
    start = time.time()
    text = client.models.generate_content(model=MODEL, contents=prompt).text
    print(f"  (Gemini call took {time.time() - start:.0f}s)")
    cache[prompt] = text
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=1)
    return text

def ask(question):
    sql_prompt = f"""You write DuckDB SQL.
{SCHEMA}
Write one SQL query that answers the question. Return only the SQL. No explanation, no code fences.
Question: {question}"""
    sql = ask_gemini(sql_prompt).strip()
    sql = re.sub(r"^```(sql)?|```$", "", sql, flags=re.M).strip()

    try:
        result = con.sql(sql).df()
    except Exception as e:
        return sql, None, f"SQL error: {e}"

    explain_prompt = f"""Question: {question}
Query result:
{result.head(20).to_string(index=False)}
Answer in one or two plain sentences, using only numbers that appear in the result. Do not add any other numbers."""
    answer = ask_gemini(explain_prompt).strip()
    return sql, result, answer

sql, result, answer = ask("How many applications were submitted online?")
print("SQL:", sql)
print(result)
print("Answer:", answer)

  (Gemini call took 225s)
  (Gemini call took 206s)
SQL: SELECT COUNT(*) FROM apps WHERE online = 1
   count_star()
0         20423
Answer: A total of 20423 applications were submitted online.


In [3]:

import pandas as pd

tests = [
    ("How many loan applications are there in total?", 31509),
    ("How many applications were submitted online?", 20423),
    ("What is the overall approval rate?", 54.85),
    ("What is the overall cancellation rate?", 33.21),
    ("What is the median number of days from start to outcome?", 19.07),
    ("What is the approval rate for car loans?", 51.43),
    ("How many applications had missing documents at least once?", 15003),
    ("Which move between two steps has the largest total waiting time?", "A_Cancelled"),
    ("How many applications were denied?", 3753),
    ("Which month of 2016 had the most new applications?", ["august", "2016-08"]),
]

batch_prompt = f"""You write DuckDB SQL.
{SCHEMA}
For each question below, write one SQL query that answers it.
Return a JSON list of strings: one SQL query per question, in the same order. Return only the JSON.
Questions:
""" + "\n".join(f"{i + 1}. {q}" for i, (q, _) in enumerate(tests))

raw = ask_gemini(batch_prompt)
raw = re.sub(r"^```(json)?|```$", "", raw.strip(), flags=re.M).strip()
sqls = json.loads(raw)
print("Queries received:", len(sqls))

def check(result, expected):
    if result is None:
        return False
    texts = [str(v).lower() for v in result.values.ravel()]
    nums = []
    for v in result.values.ravel():
        try:
            nums.append(float(v))
        except (TypeError, ValueError):
            pass
    for e in (expected if isinstance(expected, list) else [expected]):
        if isinstance(e, str):
            if any(e.lower() in t for t in texts):
                return True
        else:
            for n in nums:
                for candidate in (n, n * 100):   # accept 0.5485 or 54.85
                    if abs(candidate - e) <= max(0.005 * abs(e), 0.01):
                        return True
    return False

rows = []
for (q, expected), sql in zip(tests, sqls):
    try:
        res, err = con.sql(sql).df(), ""
    except Exception as ex:
        res, err = None, str(ex)[:80]
    shown = err if res is None else res.head(3).to_string(index=False).replace("\n", " | ")
    rows.append({"question": q[:45], "expected": expected, "got": shown[:60], "correct": check(res, expected)})

report = pd.DataFrame(rows)
print(report.to_string())
print("\nAccuracy:", report["correct"].sum(), "of", len(report))

  (Gemini call took 48s)
Queries received: 10
                                        question           expected                                                           got  correct
0  How many loan applications are there in total              31509                                  count_star() |         31509     True
1   How many applications were submitted online?              20423                                  count_star() |         20423     True
2             What is the overall approval rate?              54.85   avg(CASE  WHEN ((outcome = 'A_Pending')) THEN (1.0) ELSE 0.     True
3         What is the overall cancellation rate?              33.21   avg(CASE  WHEN ((outcome = 'A_Cancelled')) THEN (1.0) ELSE      True
4  What is the median number of days from start               19.07            median(days_to_outcome) |                    19.07     True
5       What is the approval rate for car loans?              51.43   avg(CASE  WHEN ((outcome = 'A_Pending')) THEN (1.0

In [4]:
# Fixes: Q9 expected is 3,752 applications (not 3,753 events).
# Counts (whole numbers) must match exactly. Rates allow small rounding.
# Month accepts 8 as well as "august" / "2016-08".
tests[8] = ("How many applications were denied?", 3752)
tests[9] = ("Which month of 2016 had the most new applications?", ["august", "2016-08", 8])

def check(result, expected):
    if result is None:
        return False
    values = result.values.ravel()
    texts = [str(v).lower() for v in values]
    nums = []
    for v in values:
        try:
            nums.append(float(v))
        except (TypeError, ValueError):
            pass
    for e in (expected if isinstance(expected, list) else [expected]):
        if isinstance(e, str):
            if any(e.lower() in t for t in texts):
                return True
        elif isinstance(e, int):
            if any(n == e for n in nums):          # counts: exact
                return True
        else:
            for n in nums:
                for c in (n, n * 100):             # rates: 0.5485 or 54.85
                    if abs(c - e) <= 0.01 * max(1, abs(e)) * 0.05:
                        return True
    return False

rows = []
for (q, expected), sql in zip(tests, sqls):
    try:
        res = con.sql(sql).df()
    except Exception:
        res = None
    rows.append({"question": q[:45], "expected": expected, "correct": check(res, expected)})

report = pd.DataFrame(rows)
print(report.to_string())
print("\nAccuracy:", report["correct"].sum(), "of", len(report))

                                        question              expected  correct
0  How many loan applications are there in total                 31509     True
1   How many applications were submitted online?                 20423     True
2             What is the overall approval rate?                 54.85     True
3         What is the overall cancellation rate?                 33.21     True
4  What is the median number of days from start                  19.07     True
5       What is the approval rate for car loans?                 51.43     True
6  How many applications had missing documents a                 15003     True
7  Which move between two steps has the largest            A_Cancelled     True
8             How many applications were denied?                  3752     True
9  Which month of 2016 had the most new applicat  [august, 2016-08, 8]     True

Accuracy: 10 of 10


In [5]:
baseline_prompt = """Answer each question about the BPI Challenge 2017 loan application dataset from a Dutch bank.
Give a short answer with a number for each. If you don't know, say "I don't know".
""" + "\n".join(f"{i + 1}. {q}" for i, (q, _) in enumerate(tests))

baseline = ask_gemini(baseline_prompt)
print(baseline)

  (Gemini call took 144s)
1. 31,509
2. I don't know
3. Approximately 54.7% (17,228 applications)
4. Approximately 33.1% (10,431 applications)
5. I don't know
6. I don't know
7. I don't know
8. I don't know
9. 3,753
10. I don't know


## Testing the AI assistant

The idea is simple. A manager types a question in plain English, Gemini turns it into SQL, DuckDB runs the query on the real data, and Gemini sums up the result in a sentence. Gemini never supplies a number itself. Every figure comes from the query.

To test it, I wrote 10 questions where I already knew the answer from my earlier analysis, and sent them all to Gemini in one call to save free-tier quota. Gemini's queries got all 10 right.

The test caught two mistakes of my own along the way. My checker didn't accept "8" as August, so it marked a correct answer as wrong. And I'd written down 3,753 denied applications, but the real figure is 3,752. One application had its outcome logged twice, which I'd noticed back in the data quality step and then forgot. Gemini's query counted applications correctly. I also tightened the checker so that counts have to match exactly.

For a baseline, I asked Gemini the same questions with no data at all. It got 1 out of 10. To its credit, it mostly said "I don't know" rather than inventing numbers. But three of its answers were close and still wrong. It seems to remember some published figures for this public dataset, but it doesn't know my rules, like leaving out the 98 applications that never finished. So 54.7% looks plausible, but the correct figure is 54.85%.

Ten simple questions isn't a big test. It shows the approach works for straightforward questions. I haven't tested harder ones that need joins or several steps. The free tier was also slow, with some calls taking over three minutes, which is why I cached every response.